# Multivariate forecasting: VAR and VECM

Earlier phases tested univariate baselines (LOCF, drift, seasonal naive), ARIMA/ETS,
regularised regressions and tree models, and probabilistic methods. None of them
models the **joint dynamics** of the price series. This notebook tests two
multivariate time-series families:

- **VAR** on log-differences (monthly log returns). It captures spillovers between
  materials, FX and energy but assumes no long-run relationship.
- **VECM** on log levels. It adds error-correction terms if the Johansen test finds
  cointegration, meaning prices that share a long-run equilibrium (for example
  copper and aluminium both tracking industrial demand).

Both are compared with **LOCF** (a random walk in log prices), the repo's mandatory
benchmark, on rolling origins with an expanding window.

**Two refinements over a first pass**
- **Fixed rank-1 VECM.** The Johansen-selected rank flips between origins, so we also
  test a VECM with the cointegration rank fixed at 1.
- **Publication lags.** Every backtest runs twice: with no lag (the last observation
  is the decision month) and with the repo's documented publication lags
  (`PHASE3_LAGS`), so the effective horizon is longer, as in Phases 3-5.

**Simplifications to be aware of**
- Under publication lags, each system is truncated to the last month at which **all**
  its series are published (the ragged edge is dropped). This is conservative. The
  lags are the repo's documented assumptions, not verified release dates, and
  revisions are ignored.
- The sample is short (about 120 monthly points), so systems stay small (4
  variables) and lag orders are capped.
- Errors are in **log points** (about percent error), so they are comparable across
  materials with different units.
- This is an exploratory test of model class, not a replacement for the frozen
  Phase 4 policy.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_coint_rank

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").exists()
)
sys.path.insert(0, str(root / "src"))

from schneider_electric_forecasting import _01_ingestion as ingestion  # noqa: E402
from schneider_electric_forecasting._04_features import (  # noqa: E402
    PHASE3_LAGS,
    snapshot,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Data and system definitions

All series are monthly and taken from the RM and energy workbooks through the
project's own loaders. Series are modelled in **logs**. We define small systems of
economically linked series rather than one large system, because a 5-variable VAR
with 2 lags already has about 55 coefficients for ~100 observations.

In [ ]:
rm = ingestion.load_rm_monthly_long()
en = ingestion.load_energy_monthly_long()
panel = (
    pd.concat(
        [rm[["month", "series_id", "value"]], en[["month", "series_id", "value"]]]
    )
    .pivot(index="month", columns="series_id", values="value")
    .sort_index()
)
panel.index = pd.DatetimeIndex(panel.index).to_period("M").to_timestamp()
names = pd.concat(
    [
        rm.drop_duplicates("series_id").set_index("series_id")["material"],
        en.drop_duplicates("series_id").set_index("series_id")["energy"],
    ]
).to_dict()

SYSTEMS = {
    "metals": ["S01", "S02", "S03", "S05"],  # copper, aluminium, nickel, zinc (LME)
    "steel_chain": ["S09", "S04", "E05", "E08"],  # EU steel PPI, iron ore, gas, coal
    "metals_fx_oil": ["S01", "S02", "S06", "E01"],  # copper, aluminium, EUR/USD, Brent
}
for k, v in SYSTEMS.items():
    print(k, "->", [f"{s} ({names[s]})" for s in v])

logs = np.log(panel[sorted({s for v in SYSTEMS.values() for s in v})])
print(
    "\nObservations:",
    logs.dropna().shape[0],
    "|",
    logs.dropna().index.min().date(),
    "to",
    logs.dropna().index.max().date(),
)

## 2. Pre-tests: stationarity and cointegration

A VAR in differences is valid if each series is I(1) (non-stationary in levels,
stationary in differences). A VECM adds value only if the Johansen trace test finds
cointegration. We check both on the **full sample**. This is for description only;
the backtest re-tests inside each training window, so no future data leaks into it.

In [ ]:
rows = []
for s in logs.columns:
    x = logs[s].dropna()
    rows.append(
        {
            "series": s,
            "name": names[s],
            "ADF p (level)": adfuller(x, regression="c", autolag="AIC")[1],
            "ADF p (diff)": adfuller(x.diff().dropna(), regression="c", autolag="AIC")[
                1
            ],
        }
    )
unit_roots = pd.DataFrame(rows).set_index("series")
unit_roots["I(1)?"] = (unit_roots["ADF p (level)"] > 0.05) & (
    unit_roots["ADF p (diff)"] < 0.05
)
unit_roots.round(3)

In [ ]:
rows = []
for system, cols in SYSTEMS.items():
    data = logs[cols].dropna()
    res = coint_johansen(data, det_order=0, k_ar_diff=2)
    for r in range(len(cols)):
        rows.append(
            {
                "system": system,
                "H0: rank <=": r,
                "trace stat": res.lr1[r],
                "5% crit": res.cvt[r, 1],
                "reject H0": res.lr1[r] > res.cvt[r, 1],
            }
        )
johansen = pd.DataFrame(rows)
print("Johansen trace test (constant, 2 lagged differences)")
johansen.round(2)

Read the table as: if "rank <= 0" is rejected, there is at least one cointegrating
relation. If it is **not** rejected, a VECM has nothing to add over a VAR in
differences. The "auto" VECM below falls back to a VAR in differences at rank 0.

## 3. Publication lags

The repo's `snapshot` function decides what was known at each decision date. For
IMF/Eurostat series (documented lag 2) it returns data only up to **three** months
before the decision month; for the ECB FX series (lag 1), two months. We take each
series' offset as `lag + 1` from `PHASE3_LAGS` and verify it against `snapshot`.

In [ ]:
OFFSET = {s: PHASE3_LAGS[s].maximum_lag_months + 1 for s in logs.columns}

obs_all = pd.concat([rm, en])
cat_all = pd.concat(
    [ingestion.load_rm_source_catalog(), ingestion.load_energy_source_catalog()]
)
snap = snapshot(obs_all, cat_all, "2026-09")
latest = snap.groupby("series_id")["month"].max().loc[list(OFFSET)].dt.to_period("M")
expected = pd.Series({s: pd.Period("2026-09", "M") - o for s, o in OFFSET.items()})
assert (latest == expected).all(), (latest, expected)

SYSTEM_OFFSET = {k: max(OFFSET[s] for s in v) for k, v in SYSTEMS.items()}
print("Offset (months) by series:", OFFSET)
print("System offset (ragged edge dropped):", SYSTEM_OFFSET)

## 4. Models

All models forecast log levels for 1 to `H` steps from the last observation in the
training window.

- **VAR(p)** on log-differences. `p` is chosen by BIC from 1 to 3 inside every
  training window. Forecasts are accumulated back to log levels.
- **VECM (auto)**: constant inside the cointegrating relation, 1 lagged difference,
  rank chosen by the Johansen trace test (5%) in each training window and capped at
  `n - 1`. At rank 0 it uses a VAR(1) in differences.
- **VECM (rank 1)**: the same model with the rank fixed at 1. It imposes exactly one
  long-run relationship and does not depend on test noise.
- **LOCF**: the benchmark, the last observed log price carried forward.

In [ ]:
HORIZONS = (3, 6, 9, 12)
MIN_TRAIN = 48  # training months required at the first origin
VAR_MAX_LAG = 3


def forecast_locf(train, steps):
    """Carry the last observed log price forward."""
    return pd.DataFrame(
        np.tile(train.iloc[-1].to_numpy(), (steps, 1)), columns=train.columns
    )


def forecast_var(train, steps):
    """VAR on log-differences; returns log-level forecasts and the BIC lag."""
    diffs = train.diff().dropna()
    order = VAR(diffs).select_order(maxlags=VAR_MAX_LAG).bic
    lag = max(int(order), 1)
    fit = VAR(diffs).fit(lag)
    delta = fit.forecast(diffs.to_numpy()[-lag:], steps=steps)
    levels = train.iloc[-1].to_numpy() + np.cumsum(delta, axis=0)
    return pd.DataFrame(levels, columns=train.columns), lag


def forecast_vecm(train, steps, rank=None):
    """VECM on log levels; rank=None selects it by the Johansen trace test."""
    n = train.shape[1]
    if rank is None:
        rank = select_coint_rank(
            train, det_order=0, k_ar_diff=1, method="trace", signif=0.05
        ).rank
    rank = min(rank, n - 1)
    if rank == 0:
        diffs = train.diff().dropna()
        fit = VAR(diffs).fit(1)
        delta = fit.forecast(diffs.to_numpy()[-1:], steps=steps)
        levels = train.iloc[-1].to_numpy() + np.cumsum(delta, axis=0)
        return pd.DataFrame(levels, columns=train.columns), 0
    fit = VECM(train, k_ar_diff=1, coint_rank=rank, deterministic="co").fit()
    return pd.DataFrame(fit.predict(steps=steps), columns=train.columns), rank

## 5. Rolling-origin backtest, with and without publication lags

For every **decision month** and each setting:

- *no lag*: fit on data up to the decision month and forecast `h` steps ahead.
- *publication lag*: fit on data up to `decision month - offset` (the last month
  published by then) and forecast `h + offset` steps ahead.

The **target month is identical** in both settings, so they are comparable: the
lagged setting simply works with staler information, as in Phases 3-5. Failed fits
are excluded for all models at that origin, so every model is scored on identical
windows, and both settings use the same decision months.

In [ ]:
def run_backtest(lagged):
    """Backtest all systems; returns forecast records, diagnostics and failures."""
    setting = "publication lag" if lagged else "no lag"
    records, diagnostics, failures = [], [], []
    for system, cols in SYSTEMS.items():
        data = logs[cols].dropna()
        offset = SYSTEM_OFFSET[system] if lagged else 0
        first = MIN_TRAIN + max(SYSTEM_OFFSET.values()) - 1  # same origins in both
        for origin in data.index[first : -min(HORIZONS)]:
            train = data.loc[: origin - pd.DateOffset(months=offset)]
            steps = max(HORIZONS) + offset
            try:
                var_fc, var_lag = forecast_var(train, steps)
                vecm_fc, vecm_rank = forecast_vecm(train, steps)
                fc = {
                    "LOCF": forecast_locf(train, steps),
                    "VAR": var_fc,
                    "VECM": vecm_fc,
                    "VECM_r1": forecast_vecm(train, steps, rank=1)[0],
                }
            except Exception as exc:  # numerical failure: skip origin for all models
                failures.append(
                    {"system": system, "origin": origin, "error": repr(exc)}
                )
                continue
            diagnostics.append(
                {
                    "setting": setting,
                    "system": system,
                    "origin": origin,
                    "var_lag": var_lag,
                    "vecm_rank": vecm_rank,
                }
            )
            for h in HORIZONS:
                target = origin + pd.DateOffset(months=h)
                if target not in data.index:
                    continue
                for model, frame in fc.items():
                    for series in cols:
                        records.append(
                            {
                                "setting": setting,
                                "system": system,
                                "series": series,
                                "origin": origin,
                                "h": h,
                                "model": model,
                                "forecast": frame.loc[h + offset - 1, series],
                                "actual": data.loc[target, series],
                            }
                        )
    return pd.DataFrame(records), pd.DataFrame(diagnostics), failures


outputs = [run_backtest(lagged) for lagged in (False, True)]
results = pd.concat([o[0] for o in outputs], ignore_index=True)
results["error"] = results["forecast"] - results["actual"]  # log points
diagnostics = pd.concat([o[1] for o in outputs], ignore_index=True)
print(f"{len(results):,} forecasts | failed fits: {sum(len(o[2]) for o in outputs)}")
diagnostics.groupby(["setting", "system"])["origin"].agg(["min", "max", "count"])

## 6. Results

**Relative MAE** = model MAE / LOCF MAE on the same windows and setting. Values
below 1.0 beat LOCF. The p-value is a Diebold-Mariano style test of equal
squared-error loss versus LOCF, with Newey-West (HAC) standard errors and `h` lags
because overlapping horizons make errors autocorrelated. Treat p-values as rough:
the effective sample is much smaller than the number of origins, and we run many
comparisons.

In [ ]:
def dm_pvalue(loss_diff, h):
    """HAC p-value for zero mean loss differential versus LOCF."""
    x = np.asarray(loss_diff, dtype=float)
    if len(x) < 10 or np.allclose(x, 0):
        return np.nan
    fit = sm.OLS(x, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": h})
    return fit.pvalues[0]


rows = []
for (setting, system, series, h), grp in results.groupby(
    ["setting", "system", "series", "h"]
):
    wide = grp.pivot(index="origin", columns="model", values="error")
    abs_err = wide.abs()
    locf_mae = abs_err["LOCF"].mean()
    for model in ("VAR", "VECM", "VECM_r1"):
        rows.append(
            {
                "setting": setting,
                "system": system,
                "series": series,
                "name": names[series],
                "h": h,
                "model": model,
                "n": len(wide),
                "LOCF MAE": locf_mae,
                "MAE": abs_err[model].mean(),
                "rel MAE": abs_err[model].mean() / locf_mae,
                "win rate": (abs_err[model] < abs_err["LOCF"]).mean(),
                "DM p": dm_pvalue(wide[model] ** 2 - wide["LOCF"] ** 2, h),
            }
        )
scores = pd.DataFrame(rows)

lag_cost = (
    scores.drop_duplicates(["setting", "system", "series", "h"])
    .pivot_table(index="h", columns="setting", values="LOCF MAE")
    .assign(ratio=lambda d: d["publication lag"] / d["no lag"])
)
print("Mean LOCF MAE (log points) by horizon: the cost of stale information")
lag_cost.round(3)

In [ ]:
rel = scores.pivot_table(
    index=["system", "series", "name"],
    columns=["setting", "model", "h"],
    values="rel MAE",
)
print("Relative MAE vs LOCF, WITH publication lags (below 1.00 = better than LOCF)")
rel["publication lag"].round(2)

In [ ]:
print("Relative MAE vs LOCF, NO lag (for comparison)")
rel["no lag"].round(2)

In [ ]:
lagged_rel = rel["publication lag"]
models_plot = ("VAR", "VECM", "VECM_r1")
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, model in zip(axes, models_plot, strict=True):
    mat = lagged_rel[model]
    im = ax.imshow(mat.to_numpy(), cmap="RdBu_r", vmin=0.7, vmax=1.3, aspect="auto")
    ax.set_xticks(range(len(mat.columns)), [f"h={h}" for h in mat.columns])
    ax.set_yticks(range(len(mat)), [f"{sy}: {s} {n}" for sy, s, n in mat.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{model}: relative MAE vs LOCF\n(publication lags)")
fig.colorbar(im, ax=axes, label="relative MAE (blue = better than LOCF)", shrink=0.8)
plt.show()

In [ ]:
summary = (
    scores.groupby(["setting", "model", "h"])
    .agg(
        median_rel_MAE=("rel MAE", "median"),
        share_series_beating_LOCF=("rel MAE", lambda x: (x < 1).mean()),
        share_significant_5pct=("DM p", lambda x: (x < 0.05).mean()),
    )
    .round(3)
)
summary

## 7. Did the gains survive publication lags?

The two earlier leads were the EU steel PPI (S09) and coal (E08). Here is how their
relative MAE moves when the information is stale.

In [ ]:
leads = scores[scores["series"].isin(["S09", "E08"])].pivot_table(
    index=["series", "name", "model"], columns=["setting", "h"], values="rel MAE"
)
print("Relative MAE vs LOCF")
display(leads.round(2))
pvals = scores[
    scores["series"].isin(["S09", "E08"]) & (scores["setting"] == "publication lag")
].pivot_table(index=["series", "model"], columns="h", values="DM p")
print("Diebold-Mariano p-values vs LOCF, publication lags (rough, overlapping errors)")
pvals.round(3)

## 8. What the models chose

How often did the Johansen test find cointegration inside training windows, and
what VAR lag did BIC choose? If the selected rank often differs from 1, the
rank-1 VECM is imposing a relationship the data do not always support.

In [ ]:
print("Share of origins by chosen auto-VECM cointegration rank (publication lags)")
chosen = diagnostics[diagnostics["setting"] == "publication lag"]
print(
    chosen.groupby("system")["vecm_rank"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(2)
)
print("\nShare of origins by chosen VAR lag")
print(
    chosen.groupby("system")["var_lag"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(2)
)

## 9. Do prices move together? Granger causality and impulse responses

Beyond forecasting accuracy, a VAR is useful for understanding spillovers. We fit
one VAR on the full sample (descriptive only) for the metals system, test whether
each series helps predict the others, and plot copper's response to shocks.

In [ ]:
cols = SYSTEMS["metals"]
diffs = logs[cols].dropna().diff().dropna()
fit = VAR(diffs).fit(1)
granger = pd.DataFrame(
    {
        caused: {
            causing: fit.test_causality(caused, [causing], kind="f").pvalue
            for causing in cols
            if causing != caused
        }
        for caused in cols
    }
).round(3)
granger.index.name = "causing (row) -> caused (column), p-value"
granger

In [ ]:
irf = fit.irf(12)
fig = irf.plot(orth=True, impulse="S02", response="S01", figsize=(6, 3.5))
plt.show()

## 10. Forecasts for the September 2026 decision

The actual decision date is September 30, 2026, so the last data we may use is
June 2026 (July for FX), and the effective horizons are `h + 3` steps. Shown as
the % change from the **last available** observation, for the repo's three target
months. Only the VAR and the rank-1 VECM are shown, and both should be read with the
backtest in mind: neither is shown to beat LOCF reliably.

In [ ]:
target_months = {3: "Dec 2026", 6: "Mar 2027", 9: "Jun 2027"}
decision = pd.Timestamp("2026-09-01")
rows = []
for system, cols in SYSTEMS.items():
    offset = SYSTEM_OFFSET[system]
    train = logs[cols].dropna().loc[: decision - pd.DateOffset(months=offset)]
    steps = max(target_months) + offset
    var_fc, _ = forecast_var(train, steps)
    vecm_fc, _ = forecast_vecm(train, steps, rank=1)
    for h, label in target_months.items():
        for s in cols:
            last = train[s].iloc[-1]
            rows.append(
                {
                    "system": system,
                    "series": s,
                    "name": names[s],
                    "target": label,
                    "last available": train.index[-1].strftime("%b %Y"),
                    "VAR %": 100 * (np.exp(var_fc.loc[h + offset - 1, s] - last) - 1),
                    "VECM r1 %": 100
                    * (np.exp(vecm_fc.loc[h + offset - 1, s] - last) - 1),
                }
            )
pd.DataFrame(rows).round(1)

## 11. Takeaways

Numeric results are deliberately not stored in this notebook, because they are computed
from confidential price data. Run it to regenerate them.

What to read from the outputs:

1. **Publication lags.** Compare the LOCF error with and without lags: stale
   information raises the benchmark's own error, so a model must beat a benchmark that
   is itself forecasting from older data.
2. **Rank.** Compare the auto-rank and fixed rank-1 VECM, and check how often the
   Johansen test finds rank 1 across origins before imposing a long-run relation.
3. **Individual leads.** Treat a series that beats LOCF (for example coal) as a lead to
   verify on new data, because it is one of many series examined.
4. **Proxy gains.** Check whether gains seen without lags survive once lags are applied.
5. **Conclusion.** Judge whether VAR/VECM is a credible replacement for LOCF from the
   relative MAE and DM p-values; the repo's earlier phases found no model that reliably
   beats LOCF. Do not use the September 2026 forecasts in Section 10 instead of LOCF
   unless the backtest supports it.

**Possible next steps:** a shrinkage (Bayesian) VAR for the short sample;
forecast combinations of LOCF and VAR (a small weight on a noisy model often
helps); and testing coal on a longer history, if one is available.
